In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

target_crs = "EPSG:4326"

datasets = Path("/nas/cee-ice/cjgleason/data")

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/")
metadata_dir = save_dir / "metadata" 
metadata_dir.mkdir(exist_ok=True)

In [2]:
import global_gauges as gg
facade = gg.GaugeDataFacade(providers=['usgs','eccc','brana','ukea','eauf','abom'])
sites = facade.get_stations_n_days(365*10).to_crs(target_crs)

/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/global_gauges/facade.py:208: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(provider_info)


In [3]:
csv_files = list(Path('/nas/cee-water/cjgleason/ted/gauge-match/data').glob('*.csv'))
matches = pd.concat([pd.read_csv(f)[['site_id','COMID']] for f in csv_files]) # dont need other data
matches['COMID'] = pd.to_numeric(matches['COMID'], errors='coerce')
matches.dropna(inplace=True)

matches

,site_id,COMID
0,BRANA-12160000,62054100.0
1,BRANA-12420000,62070024.0
2,BRANA-14380000,62011372.0
3,BRANA-14489000,62000288.0
5,BRANA-15040000,62048834.0
...,...,...
11109,USGS-02458600,73012694.0
11110,USGS-02460500,73012694.0
11111,USGS-09113500,77015788.0
11112,USGS-09113980,77015788.0


In [4]:
matched_sites = sites.merge(matches, left_index=True, right_on='site_id').reset_index(drop=True)
matched_sites

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider,site_id,COMID
0,La rivière Bras David à Petit-Bourg [Prise d'e...,NaN,True,16.199137,-61.666436,2025-09-09,2017-03-09,2025-08-31,0.6180,267.999,5.229478,1121.0,None,POINT (-61.66644 16.19914),eauf,EAUF-10160001,76004387.0
1,La Grande Rivière à Goyaves à Sainte-Rose [La ...,NaN,True,16.279712,-61.668130,2025-09-09,1973-03-21,2025-08-31,0.7400,280.088,6.697796,4686.0,None,POINT (-61.66813 16.27971),eauf,EAUF-10250001,76004387.0
2,La rivière des Pères à Baillif,NaN,True,16.010760,-61.739270,2025-09-09,1983-04-29,2025-09-01,0.0380,37.724,2.024348,7643.0,None,POINT (-61.73927 16.01076),eauf,EAUF-12120001,76004383.0
3,La Grande Rivière de Vieux-Habitants à Vieux-H...,NaN,True,16.086609,-61.725164,2025-09-09,1980-04-15,2025-08-31,0.3580,72.013,2.755552,10907.0,None,POINT (-61.72516 16.08661),eauf,EAUF-12210001,76004385.0
4,La rivière du Lorrain au Lorrain [SCNA],NaN,True,14.788162,-61.055505,2025-09-09,2011-02-09,2025-09-08,0.2980,34.989,2.505703,5061.0,None,POINT (-61.0555 14.78816),eauf,EAUF-22050001,61000007.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12111,PASSO DAS PEDRAS,6920.0,False,-32.519400,-53.455800,2025-11-06,2015-01-01,2025-08-31,1.5080,2789.540,129.902179,3799.0,None,POINT (-53.4558 -32.5194),brana,BRANA-88260000,64072596.0
12112,CERRO CHATO,1050.0,False,-31.864700,-53.268300,2025-11-06,1976-08-01,2025-06-30,0.3332,944.923,22.461594,16830.0,None,POINT (-53.2683 -31.8647),brana,BRANA-88575000,64068088.0
12113,PEDRO OSÓRIO,4700.0,False,-31.863300,-52.816100,2025-11-06,2000-04-19,2025-08-31,0.4346,7479.060,94.064795,9112.0,None,POINT (-52.8161 -31.8633),brana,BRANA-88641000,64067719.0
12114,PASSO DOS CARROS,131.0,False,-31.713900,-52.476700,2025-11-06,1964-10-15,2025-06-30,0.0028,176.979,3.001714,21711.0,None,POINT (-52.4767 -31.7139),brana,BRANA-88750000,64069237.0


In [5]:
merit_dir = Path("/nas/cee-ice/data/MERIT")
merit_gdfs = []
for b in tqdm(range(1,10), desc="Loading MERIT files"):
    filename = merit_dir / f"riv_pfaf_{b}_MERIT_Hydro_v07_Basins_v01_bugfix1.shp"
    merit_gdfs.append(gpd.read_file(filename))
print("Concatenating MERIT dataframes")
merit = gpd.GeoDataFrame(pd.concat(merit_gdfs))
merit.set_index('COMID', inplace=True)

Loading MERIT files: 100%|██████████| 9/9 [01:15<00:00,  8.36s/it]


Concatenating MERIT dataframes


In [7]:
from shapely.ops import nearest_points
from shapely.geometry import Point, LineString

def orient_line_upstream_downstream(comid):
    """
    Returns a LineString oriented so that the start is upstream
    and the end points downstream (towards nextdown).
    """
    line = merit.loc[comid, 'geometry']
    nextdown = merit.loc[comid, 'NextDownID']
    
    # If no downstream, just return as-is
    if nextdown not in merit.index or line is None or line.is_empty:
        return line

    ds_line = merit.loc[nextdown, 'geometry']
    if ds_line is None or ds_line.is_empty:
        return line
    
    # Convert start/end coords to Points
    start_pt = Point(line.coords[0])
    end_pt = Point(line.coords[-1])
    ds_start_pt = Point(ds_line.coords[0])
    
    # Compare distances to downstream start
    if start_pt.distance(ds_start_pt) < end_pt.distance(ds_start_pt):
        # First point is downstream, so reverse
        return LineString(line.coords[::-1])
    
    return line

def get_nearest_point_with_frac(row, min_frac=0.01, max_frac=0.99):
    comid = int(row["COMID"])
    line = orient_line_upstream_downstream(comid)
    if line is None or line.is_empty:
        return None, None  # return None for both geometry and fraction
    
    site_point = row.geometry
    dist = line.project(site_point)
    total_length = line.length
    if total_length == 0:
        return None, None
    
    frac = dist / total_length
    frac = max(min_frac, min(max_frac, frac))
    
    snapped_point = line.interpolate(frac * total_length)
    return snapped_point, frac

snapped_sites = matched_sites.copy()

results = snapped_sites.apply(get_nearest_point_with_frac, axis=1)
# Split into geometry and fraction
snapped_sites['geometry'] = results.map(lambda x: x[0])
snapped_sites['position'] = results.map(lambda x: x[1])

In [8]:
import networkx as nx

# build a MERIT digraph
G = nx.DiGraph()
G.add_edges_from(merit.loc[merit.NextDownID != 0, "NextDownID"].items())

In [9]:
# Lookup COMID -> geometry
comid_to_geom = merit.geometry.to_dict()

# Furthest downstream site (largest position) per COMID
downstream_site_per_comid = (
    snapped_sites.sort_values("position", ascending=False)
    .groupby("COMID", group_keys=False)
    .tail(1)
    .set_index("COMID")
)

# Mapping: COMID → terminal site on that reach
comid_to_terminal_site = downstream_site_per_comid["site_id"].to_dict()

In [10]:
snapped_sites

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider,site_id,COMID,position
0,La rivière Bras David à Petit-Bourg [Prise d'e...,NaN,True,16.199137,-61.666436,2025-09-09,2017-03-09,2025-08-31,0.6180,267.999,5.229478,1121.0,None,POINT (-61.66615 16.19885),eauf,EAUF-10160001,76004387.0,0.913997
1,La Grande Rivière à Goyaves à Sainte-Rose [La ...,NaN,True,16.279712,-61.668130,2025-09-09,1973-03-21,2025-08-31,0.7400,280.088,6.697796,4686.0,None,POINT (-61.66796 16.27954),eauf,EAUF-10250001,76004387.0,0.390817
2,La rivière des Pères à Baillif,NaN,True,16.010760,-61.739270,2025-09-09,1983-04-29,2025-09-01,0.0380,37.724,2.024348,7643.0,None,POINT (-61.73927 16.01),eauf,EAUF-12120001,76004383.0,0.462555
3,La Grande Rivière de Vieux-Habitants à Vieux-H...,NaN,True,16.086609,-61.725164,2025-09-09,1980-04-15,2025-08-31,0.3580,72.013,2.755552,10907.0,None,POINT (-61.73286 16.06881),eauf,EAUF-12210001,76004385.0,0.990000
4,La rivière du Lorrain au Lorrain [SCNA],NaN,True,14.788162,-61.055505,2025-09-09,2011-02-09,2025-09-08,0.2980,34.989,2.505703,5061.0,None,POINT (-61.05534 14.788),eauf,EAUF-22050001,61000007.0,0.871666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12111,PASSO DAS PEDRAS,6920.0,False,-32.519400,-53.455800,2025-11-06,2015-01-01,2025-08-31,1.5080,2789.540,129.902179,3799.0,None,POINT (-53.45833 -32.5194),brana,BRANA-88260000,64072596.0,0.329889
12112,CERRO CHATO,1050.0,False,-31.864700,-53.268300,2025-11-06,1976-08-01,2025-06-30,0.3332,944.923,22.461594,16830.0,None,POINT (-53.26817 -31.86483),brana,BRANA-88575000,64068088.0,0.021428
12113,PEDRO OSÓRIO,4700.0,False,-31.863300,-52.816100,2025-11-06,2000-04-19,2025-08-31,0.4346,7479.060,94.064795,9112.0,None,POINT (-52.81583 -31.86083),brana,BRANA-88641000,64067719.0,0.289136
12114,PASSO DOS CARROS,131.0,False,-31.713900,-52.476700,2025-11-06,1964-10-15,2025-06-30,0.0028,176.979,3.001714,21711.0,None,POINT (-52.4764 -31.7136),brana,BRANA-88750000,64069237.0,0.088306


In [11]:
# dict{COMID: NextDownID}
comid_to_down = merit["NextDownID"].to_dict()

def find_terminal_site_for_comid(comid):
    visited = set()
    while comid and comid not in visited:
        visited.add(comid)
        # If this COMID has a site, that's our terminal
        if comid in comid_to_terminal_site:
            return comid_to_terminal_site[comid]
        # move downstream
        comid = comid_to_down.get(comid, 0)
        if comid == 0:
            return None
    return None

snapped_sites["terminal_site_id"] = snapped_sites["COMID"].apply(find_terminal_site_for_comid)


In [12]:
# lookup: COMID → site_id (the furthest downstream one)
comid_to_site = downstream_site_per_comid["site_id"].to_dict()

comid_to_down = merit["NextDownID"].to_dict()

def find_terminal_site(comid):
    visited = set()
    current = comid
    terminal = None

    while current not in visited and current != 0:
        visited.add(current)

        # move downstream
        next_down = comid_to_down.get(current, 0)

        # if next reach has a site, that’s downstream gauge candidate
        if next_down in comid_to_site:
            terminal = comid_to_site[next_down]
            # but we keep going—maybe there's another gauged site further down
            current = next_down
            continue

        # if next_down == 0, we’ve reached river mouth or no further reach
        if next_down == 0:
            break

        current = next_down

    # if we found any downstream gauge, return its site_id
    if terminal:
        return terminal
        
    # else this site is terminal for its network
    return comid_to_site.get(comid)

snapped_sites["terminal_site_id"] = snapped_sites["COMID"].apply(find_terminal_site)

In [13]:
snapped_sites['site_id'].value_counts()

site_id
USGS-15200280     2
USGS-06935965     2
EAUF-10160001     1
USGS-14309000     1
USGS-14305500     1
                 ..
ABOM-154691010    1
ABOM-133885010    1
ABOM-168069010    1
ABOM-202520010    1
BRANA-88850000    1
Name: count, Length: 12114, dtype: int64

In [14]:
snapped_sites.drop_duplicates(subset='site_id', inplace=True)

In [15]:
snapped_sites['terminal_site_id'].value_counts()

terminal_site_id
USGS-07374000     1987
ECCC-02OA016       592
ABOM-213983010     488
USGS-14246900      461
EAUF-M5300010      311
                  ... 
ABOM-57711010        1
ABOM-54386010        1
ABOM-55744010        1
ABOM-773672010       1
BRANA-88850000       1
Name: count, Length: 1472, dtype: int64

In [16]:
snapped_sites['COMID'] = snapped_sites['COMID'].astype(int)

In [18]:
metadata_dir

PosixPath('/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/metadata')

In [19]:
snapped_sites.to_parquet(metadata_dir / 'gauges.parquet')

In [18]:
outlets = snapped_sites.copy()
outlets.rename(columns={'site_id':'id', 'longitude':'lng', 'latitude':'lat', 'terminal_site_id':'outlet_id'}, inplace=True)

outlets = outlets[['id','lng','lat','position','COMID','outlet_id']]
outlets.to_csv('/nas/cee-water/cjgleason/ted/graph_delineator/data/clamped_manual_matchups.csv', index=False)

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider,site_id,COMID,position,terminal_site_id
0,Ulting Sarasota,NaN,True,51.746683,0.624437,2025-09-09,2008-10-31,2025-09-07,0.006000,53.471,2.915773,5063.0,{'guid': '052d0819-2a32-47df-9b99-c243c9c8235b'},POINT (0.62429 51.74654),ukea,UKEA-037048U,23005484,0.459952,UKEA-037048U
1,Adwick,NaN,True,53.512713,-1.282505,2025-09-09,1976-01-05,2025-09-07,0.513000,72.728,3.536529,18081.0,{'guid': 'f22f80f8-1bb0-4e77-b225-291487060c6f'},POINT (-1.28261 53.51261),ukea,UKEA-F0803,23002461,0.510231,UKEA-F0908
2,Thorverton,NaN,True,50.804172,-3.511302,2025-09-09,1956-04-30,2025-09-07,0.421000,286.133,16.177839,25328.0,{'guid': '3c4d4f78-2d0e-474a-b884-65a9daca18fb'},POINT (-3.51107 50.80394),ukea,UKEA-SS90F011,23010062,0.531189,UKEA-SX99U053
3,Low Moor,NaN,True,54.488963,-1.438958,2025-09-09,1969-09-01,2025-09-07,1.570000,486.051,21.042296,20176.0,{'guid': 'd031ef9f-4e50-4c68-aa43-9b5589c32874'},POINT (-1.43854 54.48938),ukea,UKEA-F3606,23001843,0.792176,UKEA-F3606
4,Methley,NaN,True,53.726053,-1.382684,2025-09-09,1988-06-07,2025-09-07,3.179000,306.144,20.294443,13595.0,{'guid': '825e7c96-b693-4366-8e39-a7fc587d442a'},POINT (-1.38268 53.72667),ukea,UKEA-F1301,23002252,0.912074,UKEA-F1806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12111,SYLVIA GRINNELL RIVER NEAR IQALUIT,2980.0,True,63.766418,-68.580612,2025-11-07,1971-01-01,2025-10-19,0.001000,737.000,68.716181,8302.0,None,POINT (-68.58167 63.76642),eccc,ECCC-10UH001,85009406,0.327792,ECCC-10UH001
12112,APEX RIVER AT APEX,58.5,True,63.736031,-68.451561,2025-11-07,1973-06-01,2025-10-16,0.001000,15.600,1.402932,5653.0,None,POINT (-68.45167 63.73603),eccc,ECCC-10UH002,85009423,0.149674,ECCC-10UH002
12113,NORTH MILK RIVER NEAR INTERNATIONAL BOUNDARY,239.0,True,49.021961,-112.972931,2025-11-07,1950-03-01,2025-11-01,0.088000,61.500,9.978013,18903.0,None,POINT (-112.97257 49.0216),eccc,ECCC-11AA001,74000070,0.085743,USGS-07374000
12114,MILK RIVER AT MILK RIVER,2720.0,True,49.143600,-112.081703,2025-11-07,1950-01-01,2025-11-07,0.001389,222.000,9.554656,27689.0,None,POINT (-112.08197 49.14387),eccc,ECCC-11AA005,74000038,0.750409,USGS-07374000


In [19]:
outlets

,id,lng,lat,position,COMID,outlet_id
0,UKEA-037048U,0.624437,51.746683,0.459952,23005484,UKEA-037048U
1,UKEA-F0803,-1.282505,53.512713,0.510231,23002461,UKEA-F0908
2,UKEA-SS90F011,-3.511302,50.804172,0.531189,23010062,UKEA-SX99U053
3,UKEA-F3606,-1.438958,54.488963,0.792176,23001843,UKEA-F3606
4,UKEA-F1301,-1.382684,53.726053,0.912074,23002252,UKEA-F1806
...,...,...,...,...,...,...
12111,ECCC-10UH001,-68.580612,63.766418,0.327792,85009406,ECCC-10UH001
12112,ECCC-10UH002,-68.451561,63.736031,0.149674,85009423,ECCC-10UH002
12113,ECCC-11AA001,-112.972931,49.021961,0.085743,74000070,USGS-07374000
12114,ECCC-11AA005,-112.081703,49.143600,0.750409,74000038,USGS-07374000


In [39]:
gauges = outlets[outlets['outlet_id']=='EAUF-I5221010']

In [50]:
rivers = gpd.read_file("/nas/cee-ice/data/MERIT/riv_pfaf_2_MERIT_Hydro_v07_Basins_v01_bugfix1.shp").set_index('COMID')
rivers

,lengthkm,lengthdir,sinuosity,slope,uparea,order,strmDrop_t,slope_taud,NextDownID,maxup,up1,up2,up3,up4,geometry
COMID,,,,,,,,,,,,,,,
21000001,5.426344,4.292274,1.264212,0.000552,974.144905,4,3.0,0.000552,21000252,2,21000005,21000046,0,0,"LINESTRING (6.79417 47.5075, 6.79417 47.50667,..."
21000002,15.827333,6.494238,2.437135,0.000145,4367.686625,4,2.3,0.000145,21000250,2,21000004,21000047,0,0,"LINESTRING (5.685 47.53417, 5.685 47.53333, 5...."
21000003,9.735093,7.057899,1.379319,0.001876,375.243658,3,18.3,0.001876,21000251,2,21000006,21000089,0,0,"LINESTRING (5.16083 47.52583, 5.16083 47.525, ..."
21000004,3.278418,1.691680,1.937966,0.000457,4128.744138,4,1.5,0.000457,21000002,2,21000007,21000062,0,0,"LINESTRING (5.75333 47.57, 5.75417 47.57, 5.75..."
21000005,4.747953,3.711927,1.279107,0.001198,726.994686,4,5.7,0.001198,21000001,2,21000008,21000086,0,0,"LINESTRING (6.85083 47.5125, 6.85167 47.5125, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29123116,6.585529,4.768755,1.380974,0.004609,52.861319,1,30.3,0.004609,29122871,0,0,0,0,0,"LINESTRING (46.62917 13.46833, 46.62833 13.467..."
29123117,11.744647,8.943371,1.313224,0.020649,63.425966,1,241.9,0.020649,29122721,0,0,0,0,0,"LINESTRING (46.07083 13.41833, 46.07 13.41917,..."
29123118,9.366883,7.875646,1.189348,0.013146,82.745786,1,122.8,0.013146,0,0,0,0,0,0,"LINESTRING (46.2225 13.43167, 46.22167 13.4325..."


In [51]:
rivers.loc[23010242]

lengthkm                                               9.926151
lengthdir                                               5.96784
sinuosity                                              1.663274
slope                                                  0.001207
uparea                                               425.483272
order                                                         2
strmDrop_t                                                 12.0
slope_taud                                             0.001207
NextDownID                                             23010136
maxup                                                         2
up1                                                    23010265
up2                                                    23012749
up3                                                           0
up4                                                           0
geometry      LINESTRING (-1.3733333333333366 50.93583333333...
Name: 23010242, dtype: object

In [46]:
def get_network_comids(
    outlet_comid: int, rivers: gpd.GeoDataFrame) -> set:
    """
    Find all COMIDs in the watershed network containing this outlet.

    Uses recursive network traversal to find all upstream catchments from the outlet.
    """

    def addnode(B: list, node_id):
        """Recursively assemble list of upstream unit catchments."""
        B.append(node_id)
        for up_field in ["up1", "up2", "up3", "up4"]:
            up = rivers[up_field].loc[node_id]
            if up != 0:
                addnode(B, up)

    network_comids = []
    addnode(network_comids, outlet_comid)

    return network_comids


get_network_comids(23010242, rivers)

[23010242]

In [53]:
sites

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider
site_id,,,,,,,,,,,,,,,
UKEA-037048U,Ulting Sarasota,NaN,True,51.746683,0.624437,2025-09-09,2008-10-31,2025-09-07,0.006,53.471000,2.915773,5063.0,{'guid': '052d0819-2a32-47df-9b99-c243c9c8235b'},POINT (0.62444 51.74668),ukea
UKEA-510810,Beggearn Huish,NaN,True,51.146322,-3.373695,2025-09-09,1967-01-01,2025-09-07,0.026,11.051000,0.892094,20432.0,{'guid': '48513a18-e485-4317-ae92-93bf4f7f3e54'},POINT (-3.3737 51.14632),ukea
UKEA-F0803,Adwick,NaN,True,53.512713,-1.282505,2025-09-09,1976-01-05,2025-09-07,0.513,72.728000,3.536529,18081.0,{'guid': 'f22f80f8-1bb0-4e77-b225-291487060c6f'},POINT (-1.2825 53.51271),ukea
UKEA-SS90F011,Thorverton,NaN,True,50.804172,-3.511302,2025-09-09,1956-04-30,2025-09-07,0.421,286.133000,16.177839,25328.0,{'guid': '3c4d4f78-2d0e-474a-b884-65a9daca18fb'},POINT (-3.5113 50.80417),ukea
UKEA-521410,Iwood,NaN,True,51.363977,-2.788890,2025-09-09,1973-03-13,2025-09-07,0.096,16.227000,0.813153,18047.0,{'guid': '959f3e4f-bb6e-4f4a-8082-0157eea99482'},POINT (-2.78889 51.36398),ukea
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ECCC-11AE008,POPLAR RIVER AT INTERNATIONAL BOUNDARY,928.0,True,48.990280,-105.696114,2025-11-07,1950-04-10,2024-05-06,0.001,142.000000,0.784039,17491.0,None,POINT (-105.69611 48.99028),eccc
ECCC-11AE009,ROCK CREEK BELOW HORSE CREEK NEAR INTERNATIONA...,837.0,True,48.969440,-106.838890,2025-11-07,1956-09-20,2023-12-13,0.001,98.000000,0.666541,18515.0,None,POINT (-106.83889 48.96944),eccc
ECCC-11AE010,HAY MEADOW CREEK NEAR LISIEUX,187.0,True,49.270279,-105.972778,2025-11-07,1967-07-14,2022-10-31,0.001,20.200001,0.167529,13471.0,None,POINT (-105.97278 49.27028),eccc
